In [1]:
# Cargo las variables de entorno desde .env para que la API key de Open AI esté disponible
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Creo el cliente de OpenAI que voy a usar para hacer llamadas a la API
from openai import OpenAI

openai_client = OpenAI()

In [3]:
# Defino la funcion llm() para hacer llamadas al modelo sin repetir codigo cada vez
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return response.output_text

In [4]:
llm("¿Cuál es la capital de Francia?")

'La capital de Francia es París.'

In [5]:
# Descargo las lesson pages del repo del curso desde un commit fijo
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
#Reviso que tiene cada uno de los archivos descargados
files[5]

RawRepositoryFile(filename='01-agentic-rag/lessons/06-building-prompt.md', content='# Building the Prompt\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe LLM doesn\'t see our documents unless we pass them in. So we need\nto build a prompt that includes the user\'s question and the search\nresults.\n\nWhen we build AI systems, we usually split the prompt into two parts:\n\n- Instructions (also called the system prompt): this tells the LLM how\n  to behave. It never changes, so it\'s the same for every request.\n- User prompt: this changes with every request. It carries the actual\n  question and the retrieved context.\n\nWe split them because the instructions are fixed and the user prompt is\nnot. Keeping them apart makes the fixed part easy to reuse and the\nchanging part easy to build fresh each time.\n\n## Instructions\n\nThe instructions tell the LLM its role and how to answer:\n\n```python\nINSTRUCTIONS = """

In [7]:
# Parseo cada archivo descargado y armo la lista de documentos
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(len(documents))

72


There are 72 lessons pages.

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [9]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [10]:
results = index.search("How does the agentic loop keep calling the model until it stops?")

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [11]:
import sys
sys.path.insert(0, "M1 lessons")
from rag_helper import RAGBase

class LessonRAG(RAGBase):

    # Sobreescribo search() porque mi i­ndice no tiene campos 'question', 'section' ni 'course'
    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    # Sobreescribo build_context() para construir el contexto con mis campos reales
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    # Sobreescribo llm() para devolver la respuesta completa y poder leer el uso de tokens
    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        return response

    # Sobreescribo rag() para devolver tanto la respuesta como el uso de tokens
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage

In [12]:
# Instancio mi RAG con el i­ndice y el cliente de OpenAI que ya tengo en memoria
rag = LessonRAG(index=index, llm_client=openai_client)

# Corro la query de Q3 y capturo la respuesta y el uso de tokens
answer, usage = rag.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print(usage.input_tokens)

The agentic loop keeps calling the model repeatedly until it stops by running a `while True` loop that does the following each iteration:

1. It sends the current conversation messages (which include instructions, user questions, model outputs, and tool results so far) to the LLM via the Responses API, allowing the model to decide the next action.

2. It then processes the model's output messages. If any of the outputs are `function_call` entries, it:
   - Executes the corresponding tool function (e.g., `search`) using the provided arguments.
   - Appends the function call outputs back into the conversation messages.

3. It sets a flag `has_function_calls` to `True` if any function calls occurred this iteration.

4. The loop increments an iteration counter and repeats.

5. The loop condition to stop is when the model's response contains **no function calls** (`has_function_calls == False`), meaning the model has finished its reasoning and provides a final answer.

Because the full conv

In [13]:
# Divido los documentos en chunks de 2000 caracteres con paso de 1000 (queda un overlap de 1000 caracteres entre chunks)
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [14]:
# Indexo los chunks igual que los documentos: content como texto, filename como keyword
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [15]:
# Apunto el RAG al i­ndice de chunks y corro la misma query que en Q3
rag_chunks = LessonRAG(index=chunk_index, llm_client=openai_client)

answer_chunks, usage_chunks = rag_chunks.rag("How does the agentic loop keep calling the model until it stops?")

print(usage_chunks.input_tokens)

2310


Consume 3x veces menos tokens que la consulta con los documentos completos

In [16]:
def search(query: str) -> list:
    """Search the course lessons for relevant content."""
    return chunk_index.search(query, num_results=5)

In [17]:
from toyaikit.tools import Tools

tools = Tools()
tools.add_tool(search)

In [18]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient

llm_client = OpenAIClient(model="gpt-4.1-mini", client=openai_client)

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.",
    llm_client=llm_client,
    chat_interface=None,
)

In [19]:
result = runner.loop("How does the agentic loop work, and how is it different from plain RAG?")

search_calls = sum(1 for msg in result.new_messages if hasattr(msg, 'type') and msg.type == 'function_call')

print(f"Search calls: {search_calls}")
print(result.last_message)

Search calls: 3
The agentic loop is a process where an AI agent interacts in a loop by calling an LLM (Large Language Model), executing any tool calls the model returns, sending the results back to the model, and continuing this until the model produces a final answer with no more tool calls. This loop enables dynamic decision-making where the agent decides what to do next based on the goal and current context, essentially managing conversation history and actions in a multi-turn interaction.

In contrast, plain RAG (Retrieval Augmented Generation) is a simpler pipeline with three main steps:
1. Retrieve relevant documents or information from a data source using a search step.
2. Build a prompt by adding the retrieved context.
3. Generate an answer from the LLM based on the augmented prompt.

Plain RAG involves typically a single LLM call per query, while the agentic loop involves multiple calls as the model interacts with tools and updates its context dynamically.

The agentic loop di